In [ ]:
import os
import sys
import json
import copy
import time
import shutil
import importlib
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Subset

In [ ]:
BASE_DIR = os.environ.get("MELD_BASE_DIR", "/kaggle/working")
config = json.load(open(os.path.join(BASE_DIR, "config.json")))
context_config = json.load(open(os.path.join(BASE_DIR, "context_config.json")))
model_config = json.load(open(os.path.join(BASE_DIR, "model_config.json")))
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESULTS_DIR, CKPT_DIR = config["paths"]["results_dir"], config["paths"]["checkpoints_dir"]
EMOTIONS, IN_DIMS = config["emotions"], context_config["in_dims"]
MODEL_KWARGS = model_config["model_kwargs"]

def env_list(name, default, cast):
    return [cast(x) for x in os.environ.get(name, default).split(",") if x.strip()]

SEEDS = env_list("MELD_P9_SEEDS", "42,43,44,45,46", int)
LAMBDA_GRID = env_list("MELD_P9_LAMBDAS", "0.5,1.0,2.0,4.0", float)          
BATCH_SIZE = int(os.environ.get("MELD_P9_BATCH", 16))                     
FAMILY = os.environ.get("MELD_P9_FAMILY", "legacy")                        
assert FAMILY in ("legacy", "poe"), "MELD_P9_FAMILY must be 'legacy' or 'poe'"
FUSION_BASE, FUSION_REL = ("gated", "reliability_gated") if FAMILY == "legacy" else ("poe_gated", "poe_reliability")
MOD_DROPOUT = float(os.environ.get("MELD_P9_MOD_DROPOUT", 0.3))             
NOISE_AUG_P = float(os.environ.get("MELD_P9_NOISE_AUG_P", 0.2))            
DEAD_AUG_P = float(os.environ.get("MELD_P9_DEAD_AUG_P", 0.15))            
ROBUST_RHO = float(os.environ.get("MELD_P9_ROBUST_RHO", 0.5))             
MASK_INVALID = os.environ.get("MELD_P9_MASK_INVALID", "1").strip().lower() in ("1", "true", "yes")   
U_NORM = os.environ.get("MELD_P9_U_NORM", "")                              
U_CLIP = float(os.environ.get("MELD_P9_U_CLIP", 5.0))                      
U_MEASURE = os.environ.get("MELD_P9_U_MEASURE", "ent")                         
MC_SCOPE = os.environ.get("MELD_P9_MC_SCOPE", "")                           
U_HINGE = os.environ.get("MELD_P9_U_HINGE", "1").strip().lower() in ("1", "true", "yes")
RUN_VERIFICATION = os.environ.get("MELD_P9_VERIFY", "1") == "1"
LR = float(os.environ.get("MELD_P9_LR", 3e-4))
WD = float(os.environ.get("MELD_P9_WD", 1e-2))
DROPOUT = os.environ.get("MELD_P9_DROPOUT", "").strip()                      
HP = {"lr": LR, "weight_decay": WD, "max_epochs": int(os.environ.get("MELD_P9_MAX_EPOCHS", 40)),
      "patience": int(os.environ.get("MELD_P9_PATIENCE", 8)), "grad_clip": 1.0, "aux_weight": model_config["aux_weight"],
      "lr_factor": 0.5, "lr_patience": 3}                                  
if MOD_DROPOUT > 0 or NOISE_AUG_P > 0 or DEAD_AUG_P > 0:
    HP.update({"mod_dropout": MOD_DROPOUT, "noise_aug_p": NOISE_AUG_P, "dead_aug_p": DEAD_AUG_P, "noise_aug_levels": [0.5, 4.0]})
if ROBUST_RHO > 0:
    HP.update({"robust_rho": ROBUST_RHO})   
SELECT_METRIC = os.environ.get("MELD_SELECT_METRIC", "macro_f1").strip().lower()
assert SELECT_METRIC in ("macro_f1", "accuracy"), "MELD_SELECT_METRIC must be 'macro_f1' or 'accuracy'"
SEL_TAG = ""
if SELECT_METRIC != "macro_f1":
    HP["select_metric"] = SELECT_METRIC
    SEL_TAG = "_sel-" + SELECT_METRIC

LOSS_SELECTION = None
_sel_path = os.path.join(RESULTS_DIR, "loss_sweep_selected.json")
if "MELD_P9_LOSS" not in os.environ and os.path.exists(_sel_path):
    _sel = json.load(open(_sel_path))
    _now = {k: config.get(k) for k in ("audio_feature_set", "audio_model", "video_backbone", "video_hsemotion_model", "text_feature_set", "audio_ft_feature_set") if config.get(k) is not None}
    assert _sel.get("feature_desc") == _now, ("results/loss_sweep_selected.json was made for different features -- re-run Phase_09a "
                                              "(or set MELD_P9_LOSS explicitly / delete the file to ignore the sweep)")
    for _k, _v in _sel.get("env", {}).items():
        os.environ.setdefault(_k, str(_v))
    LOSS_SELECTION = {"from": "loss sweep (Phase 9a)", "selected": _sel["selected"], "reason": _sel["reason"]}
    print(f"Loss from the validation-only sweep: {_sel['selected']} -- {_sel['reason']}")
LOSS_KIND = os.environ.get("MELD_P9_LOSS", "wce").strip().lower()
assert LOSS_KIND in ("wce", "wce_soft", "focal", "logit_adj"), "MELD_P9_LOSS must be one of: wce | wce_soft | focal | logit_adj"
FOCAL_GAMMA = float(os.environ.get("MELD_P9_FOCAL_GAMMA", 2.0))
WEIGHT_POWER = float(os.environ.get("MELD_P9_WEIGHT_POWER", 0.5))     
LA_TAU = float(os.environ.get("MELD_P9_LA_TAU", 1.0))
SWEEP_TAG = os.environ.get("MELD_P9_SWEEP_TAG", "").strip()          
LOSS_TAG = ""
if LOSS_KIND == "wce_soft":
    HP["loss"] = {"kind": "wce_soft", "weight_power": WEIGHT_POWER}
    LOSS_TAG = f"_loss-wce_soft-p{WEIGHT_POWER:g}"
elif LOSS_KIND == "focal":
    HP["loss"] = {"kind": "focal", "gamma": FOCAL_GAMMA, "weight_power": WEIGHT_POWER}
    LOSS_TAG = f"_loss-focal-g{FOCAL_GAMMA:g}-p{WEIGHT_POWER:g}"
elif LOSS_KIND == "logit_adj":
    HP["loss"] = {"kind": "logit_adj", "tau": LA_TAU}
    LOSS_TAG = f"_loss-logit_adj-t{LA_TAU:g}"
OUT_DIR = RESULTS_DIR if not SWEEP_TAG else os.path.join(RESULTS_DIR, "sweeps", SWEEP_TAG)
os.makedirs(OUT_DIR, exist_ok=True)


FEAT_MASK = float(os.environ.get("MELD_P9_FEAT_MASK", 0.0))        
UTT_MASK_P = float(os.environ.get("MELD_P9_UTT_MASK_P", 0.0))      
UTT_NOISE_P = float(os.environ.get("MELD_P9_UTT_NOISE_P", 0.0))    
for _n, _v in (("MELD_P9_FEAT_MASK", FEAT_MASK), ("MELD_P9_UTT_MASK_P", UTT_MASK_P), ("MELD_P9_UTT_NOISE_P", UTT_NOISE_P)):
    assert 0.0 <= _v < 1.0, f"{_n} must be in [0, 1)"
AUG_TAG = ""
if FEAT_MASK > 0:
    HP["feat_mask_rate"] = FEAT_MASK
    AUG_TAG += f"-f{FEAT_MASK:g}"
if UTT_MASK_P > 0:
    HP["utt_mask_p"] = UTT_MASK_P
    AUG_TAG += f"-m{UTT_MASK_P:g}"
if UTT_NOISE_P > 0:
    HP["utt_noise_p"] = UTT_NOISE_P
    HP.setdefault("noise_aug_levels", [0.5, 4.0])
    AUG_TAG += f"-n{UTT_NOISE_P:g}"
AUG_TAG = ("_aug" + AUG_TAG) if AUG_TAG else ""
HP_TAG = "" if (LR == 3e-4 and WD == 1e-2 and not DROPOUT) else (f"_hp-lr{LR:.2e}-wd{WD:.1e}" + (f"-do{float(DROPOUT):g}" if DROPOUT else ""))
print(f"Device: {device} | family: {FAMILY} | seeds: {SEEDS} | lambda grid: {LAMBDA_GRID} | batch: {BATCH_SIZE}\nHP: {HP}")

In [ ]:
MODULE_SRC = r'''
import os
import json
import time
import copy
import hashlib
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix
from meld_models import ContextFusionModel, compute_loss
from meld_context import batch_to_device


def _atomic_torch_save(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)


def _atomic_json(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)


def fingerprint(cfg):
    return hashlib.sha1(json.dumps(cfg, sort_keys=True, default=str).encode()).hexdigest()[:16]


def compute_metrics(y_true, y_pred, emotions):
    C = len(emotions)
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=list(range(C)), zero_division=0)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", labels=list(range(C)), zero_division=0)),
        "per_class": {e: {"precision": float(p[i]), "recall": float(r[i]), "f1": float(f[i]), "support": int(s[i])}
                      for i, e in enumerate(emotions)},
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=list(range(C))).tolist(),
    }


@torch.no_grad()
def evaluate_model(model, loader, device, emotions, eval_seed=0):
    """Deterministic evaluation (dropout off; MC-dropout passes seeded with eval_seed without touching the training RNG).
    Returns (metrics, arrays) where arrays are aligned by CSV row index (row_idx)."""
    was_training = model.training
    model.eval()
    rows, ys, probs, us, urs, ws, wcs, mps, ce = [], [], [], [], [], [], [], [], 0.0
    devices = [torch.cuda.current_device()] if torch.cuda.is_available() else []
    with torch.random.fork_rng(devices=devices):
        torch.manual_seed(eval_seed)
        for batch in loader:
            batch = batch_to_device(batch, device)
            out = model(batch)
            sel = batch["mask"]
            logits = out["logits"][sel]
            y = batch["labels"][sel]
            ce += float(torch.nn.functional.cross_entropy(logits, y, reduction="sum"))
            rows.append(batch["row_idx"][sel].cpu().numpy())
            ys.append(y.cpu().numpy())
            probs.append(torch.softmax(logits, -1).cpu().numpy())
            us.append(out["u"][sel].cpu().numpy())
            urs.append(out["u_raw"][sel].cpu().numpy())
            mps.append(torch.stack([out["modality_logits"][m][sel].argmax(-1) for m in model.modalities], -1).cpu().numpy())
            ws.append(out["w"][sel].cpu().numpy())
            wcs.append(out["w_content"][sel].cpu().numpy())
    model.train(was_training)
    arr = {"row_idx": np.concatenate(rows), "y_true": np.concatenate(ys), "probs": np.concatenate(probs),
           "u": np.concatenate(us), "u_raw": np.concatenate(urs), "w": np.concatenate(ws),
           "w_content": np.concatenate(wcs), "mod_pred": np.concatenate(mps)}
    arr["y_pred"] = arr["probs"].argmax(1)
    order = np.argsort(arr["row_idx"], kind="stable")          
    arr = {k: v[order] for k, v in arr.items()}
    if len(np.unique(arr["row_idx"])) != len(arr["row_idx"]):
        raise RuntimeError("Duplicate rows in evaluation output")
    metrics = compute_metrics(arr["y_true"], arr["y_pred"], emotions)
    metrics["loss"] = ce / len(arr["y_true"])
    metrics["n"] = int(len(arr["y_true"]))
    return metrics, arr


def weight_diagnostics(arr, modalities):
    """How far do reliability-aware weights move from the content-gate weights, and how uniform are they?"""
    w, wc, u, ur = arr["w"], arr["w_content"], arr["u"], arr["u_raw"]
    return {
        "u_raw_mean": {m: float(ur[:, i].mean()) for i, m in enumerate(modalities)},
        "u_mean": {m: float(u[:, i].mean()) for i, m in enumerate(modalities)},          
        "u_std": {m: float(u[:, i].std()) for i, m in enumerate(modalities)},
        "w_mean": {m: float(w[:, i].mean()) for i, m in enumerate(modalities)},
        "w_content_mean": {m: float(wc[:, i].mean()) for i, m in enumerate(modalities)},
        "w_spread": float((w.max(1) - w.min(1)).mean()),                 
        "w_shift_from_content": float(np.abs(w - wc).sum(1).mean() / 2),   
    }


def save_predictions(path, arr, emotions, modalities):
    df = pd.DataFrame({"row_idx": arr["row_idx"], "y_true": arr["y_true"], "y_pred": arr["y_pred"]})
    for i, e in enumerate(emotions):
        df[f"p_{e}"] = arr["probs"][:, i]
    for i, m in enumerate(modalities):
        df[f"ur_{m}"], df[f"u_{m}"] = arr["u_raw"][:, i], arr["u"][:, i]      
        df[f"w_{m}"], df[f"wc_{m}"] = arr["w"][:, i], arr["w_content"][:, i]
        df[f"pred_{m}"] = arr["mod_pred"][:, i]                             
    df.to_csv(path, index=False)


def _rng_state():
    st = {"torch": torch.get_rng_state()}
    if torch.cuda.is_available():
        st["cuda"] = torch.cuda.get_rng_state_all()
    return st


def _set_rng_state(st):
    torch.set_rng_state(st["torch"].cpu())
    if "cuda" in st and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([s.cpu() for s in st["cuda"]])



def compute_feature_std(dataset):
    """Per-dimension std of every modality over the usable TRAIN rows (scale for noise augmentation)."""
    return {m: torch.from_numpy(np.maximum(x[dataset.valid[m]].std(axis=0), 1e-6).astype(np.float32)) for m, x in dataset.features.items()}


def augment_batch(batch, hp, feature_std, gen):   
    """Training-time corruption (CPU tensors).  Per dialogue and modality:
       - modality dropout (prob hp['mod_dropout']): the modality is flagged unusable (valid=False) for the whole dialogue
         (the model sees the missing token and, in poe modes, its gate is exactly 0). At least one modality always survives.
       - silent noise (prob hp['noise_aug_p']): Gaussian noise with std = level * train-std, level ~ U(hp['noise_aug_levels']),
         NO flag (the model has to notice the corruption itself, e.g. through its uncertainty)."""
    B, M = batch["mask"].shape[0], len(batch["features"])
    mods = list(batch["features"].keys())
    p_drop = torch.full((M,), float(hp.get("mod_dropout", 0.0)))
    for j, m in enumerate(mods):                       
        if m in hp.get("mod_dropout_per", {}):
            p_drop[j] = float(hp["mod_dropout_per"][m])
    drop = torch.rand(B, M, generator=gen) < p_drop.unsqueeze(0)
    all_dropped = drop.all(dim=1)
    if all_dropped.any():
        keep = torch.randint(0, M, (B,), generator=gen)
        drop[all_dropped, keep[all_dropped]] = False
    dead = (torch.rand(B, M, generator=gen) < hp.get("dead_aug_p", 0.0)) & ~drop     
    noise = (torch.rand(B, M, generator=gen) < hp.get("noise_aug_p", 0.0)) & ~drop & ~dead
    lo, hi = hp.get("noise_aug_levels", [0.5, 4.0])
    lvl = lo + (hi - lo) * torch.rand(B, M, generator=gen)
    valid0 = {m: batch["valid"][m].clone() for m in mods}
    for j, m in enumerate(mods):
        batch["valid"][m] = batch["valid"][m] & ~drop[:, j].unsqueeze(1)
        if dead[:, j].any():
            batch["features"][m] = batch["features"][m] * (~dead[:, j]).float().view(B, 1, 1)
        if noise[:, j].any():
            eps = torch.randn(batch["features"][m].shape, generator=gen)
            scale = (lvl[:, j] * noise[:, j].float()).view(B, 1, 1) * feature_std[m].view(1, 1, -1)
            batch["features"][m] = batch["features"][m] + scale * eps
    _old = torch.stack([valid0[m] for m in mods], dim=-1)
    _new = torch.stack([batch["valid"][m] for m in mods], dim=-1)
    _lost = _old.any(-1) & ~_new.any(-1) & batch["mask"]
    if _lost.any():
        for m in mods:
            batch["valid"][m] = batch["valid"][m] | (_lost & valid0[m])
    
    real = batch["mask"]                                                   
    L = real.shape[1]
    p_utt_noise, p_utt_mask, r_feat = float(hp.get("utt_noise_p", 0.0)), float(hp.get("utt_mask_p", 0.0)), float(hp.get("feat_mask_rate", 0.0))
    if p_utt_noise > 0:
        lo_, hi_ = hp.get("noise_aug_levels", [0.5, 4.0])
        sel = torch.rand(B, L, M, generator=gen) < p_utt_noise
        lv = torch.exp(float(np.log(lo_)) + float(np.log(hi_ / lo_)) * torch.rand(B, L, M, generator=gen))
        for j, m in enumerate(mods):
            ok = sel[:, :, j] & batch["valid"][m] & real & ~(drop[:, j] | dead[:, j] | noise[:, j]).unsqueeze(1)
            eps = torch.randn(batch["features"][m].shape, generator=gen)
            scale = (lv[:, :, j] * ok.float()).unsqueeze(-1) * feature_std[m].view(1, 1, -1)
            batch["features"][m] = batch["features"][m] + scale * eps
    if p_utt_mask > 0:
        vs = torch.stack([batch["valid"][m] for m in mods], dim=-1)                       
        new = vs & ~((torch.rand(B, L, M, generator=gen) < p_utt_mask) & real.unsqueeze(-1))
        none_left = vs.any(-1) & ~new.any(-1) & real
        keep_idx = (torch.rand(B, L, M, generator=gen) * vs.float()).argmax(-1)           
        new = new | (torch.nn.functional.one_hot(keep_idx, M).bool() & none_left.unsqueeze(-1))
        for j, m in enumerate(mods):
            batch["valid"][m] = new[:, :, j]
    if r_feat > 0:
        for m in mods:
            batch["features"][m] = batch["features"][m] * (torch.rand(batch["features"][m].shape, generator=gen) >= r_feat).float()
    return batch


def corrupt_dataset(ds, kind, modality, level=None, feature_std=None, seed=123):
    """Shallow copy of a MELDDialogueDataset with ONE modality degraded (same definitions as Phase 11):
    kind='missing' -> flagged unusable | 'dead' -> zeroed, NOT flagged | 'noise' -> + level * train_std * N(0,1), NOT flagged."""
    c = copy.copy(ds)
    c.features, c.valid = dict(ds.features), {m: v.copy() for m, v in ds.valid.items()}
    if kind == "missing":
        c.valid[modality] = np.zeros_like(ds.valid[modality])
    elif kind == "dead":
        c.features[modality] = np.zeros_like(ds.features[modality])
    elif kind == "noise":
        rs = np.random.RandomState(seed)
        std = feature_std[modality].numpy()
        c.features[modality] = (ds.features[modality] + level * std * rs.randn(*ds.features[modality].shape)).astype(np.float32)
    else:
        raise ValueError(kind)
    return c


def robust_score(clean_f1, degraded_f1, rho):
    """Model-selection score: (1 - rho) * clean macro-F1 + rho * mean macro-F1 over the degraded validation conditions.
    rho = 0 reproduces the original clean-only selection exactly."""
    if rho <= 0 or not degraded_f1:
        return float(clean_f1)
    return float((1 - rho) * clean_f1 + rho * np.mean(list(degraded_f1.values())))


LOSS_KINDS = ("wce", "wce_soft", "focal", "logit_adj")


def prepare_loss(class_weights, loss_cfg):
    """class_weights: the BALANCED inverse-frequency weights w_c = N / (C n_c) from the TRAIN split (so the class prior is pi_c ~ 1 / w_c).
    loss_cfg: {"kind": ..., "weight_power": p, "gamma": g, "tau": t}.
      wce       : weighted cross-entropy with the weights as given (identical to meld_models.compute_loss)
      wce_soft  : weights w_c ** p, renormalised so that sum_c pi_c * w_c = 1 (same overall loss scale)  [p = 1 -> wce, p = 0 -> unweighted]
      focal     : weighted focal loss  w_c * (1 - p_t) ** gamma * (-log p_t) with softened weights (default p = 0.5), normalised by sum w_y
      logit_adj : CE on (logits + tau * log pi), no class weights; inference uses the raw logits (Menon et al., logit-adjusted loss)"""
    cfg = loss_cfg or {"kind": "wce"}
    kind = cfg.get("kind", "wce")
    if kind not in LOSS_KINDS:
        raise ValueError(f"loss kind must be one of {LOSS_KINDS}, got {kind!r}")
    w = class_weights.detach().float()
    inv = 1.0 / w
    prior = inv / inv.sum()
    if abs(float((w * prior).sum()) - 1.0) > 1e-3:
        raise ValueError("class_weights must be the balanced inverse-frequency weights (sum_c pi_c * w_c = 1) for the imbalance losses")
    spec = {"kind": kind, "weights": w, "offset": None, "gamma": 0.0}
    if kind in ("wce_soft", "focal"):
        ws = w ** float(cfg.get("weight_power", 0.5))
        spec["weights"] = ws / (ws * prior).sum()
    if kind == "focal":
        spec["gamma"] = float(cfg.get("gamma", 2.0))
    if kind == "logit_adj":
        spec["weights"] = None
        spec["offset"] = float(cfg.get("tau", 1.0)) * prior.log()
    return spec


def _ce_spec(logits, y, spec):
    if spec["offset"] is not None:
        logits = logits + spec["offset"].to(logits.device)
    w = None if spec["weights"] is None else spec["weights"].to(logits.device)
    if spec["kind"] == "focal":
        logp = torch.log_softmax(logits, dim=-1).gather(1, y.unsqueeze(1)).squeeze(1)     
        wy = w[y]
        mod = (1.0 - logp.exp()).clamp(min=1e-6) ** spec["gamma"]
        return -(wy * mod * logp).sum() / wy.sum()
    return torch.nn.functional.cross_entropy(logits, y, weight=w)


def imbalance_loss(out, batch, spec, aux_weight=0.5):
    """Same structure as meld_models.compute_loss (fused loss + aux_weight * mean per-modality head loss; padded positions excluded),
    with the selected imbalance loss applied EVERYWHERE (fused prediction and every modality head)."""
    mask, y = batch["mask"], batch["labels"][batch["mask"]]
    main = _ce_spec(out["logits"][mask], y, spec)
    aux = torch.stack([_ce_spec(out["modality_logits"][m][mask], y, spec) for m in out["modality_logits"]]).mean()
    return main + aux_weight * aux, {"loss_main": float(main.detach()), "loss_aux": float(aux.detach())}


def train_run(run_dir, model_kwargs, hp, seed, train_loader, val_loader, class_weights, device, emotions,
              calib_loader=None, feature_std=None, stop_after_epoch=None, verbose=True, robust_loaders=None):
    """Train one (model, seed). Selects the checkpoint by VALIDATION MACRO-F1. Resumable.

    run_dir/ config.json, log.jsonl, last.pt (full state for resume), best.pt, val_predictions.csv, result.json
    hp keys: lr, weight_decay, max_epochs, patience, grad_clip, aux_weight, lr_factor, lr_patience
    calib_loader: TRAIN utterances in fixed order. The per-modality uncertainty statistics (u_mu, u_sigma) are calibrated on it
    before training and after every epoch (before validation), so the stored/evaluated normalisation is a train-set statistic.
    stop_after_epoch is for tests: returns status 'interrupted' after that many epochs (resume later).
    """
    os.makedirs(run_dir, exist_ok=True)
    cfg = {"model_kwargs": model_kwargs, "hp": hp, "seed": seed, "class_weights": class_weights.tolist()}
    fp = fingerprint(cfg)
    cfg_path, res_path = os.path.join(run_dir, "config.json"), os.path.join(run_dir, "result.json")
    if os.path.exists(cfg_path):
        old = json.load(open(cfg_path))
        if old["fingerprint"] != fp:
            raise RuntimeError(f"{run_dir} was produced with a different configuration -- refusing to overwrite/resume.")
    else:
        _atomic_json({"fingerprint": fp, **cfg}, cfg_path)
    if os.path.exists(res_path):
        res = json.load(open(res_path))
        if verbose:
            print(f"[skip] {run_dir} already completed (best val macro-F1 {res['best_val_macro_f1']:.4f})")
        res["status"] = "skipped_completed"
        return res

    torch.manual_seed(seed); np.random.seed(seed)
    model = ContextFusionModel(**model_kwargs).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=hp["lr"], weight_decay=hp["weight_decay"])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=hp["lr_factor"], patience=hp["lr_patience"])
    cw = class_weights.to(device)
    loss_spec = prepare_loss(cw, hp["loss"]) if hp.get("loss") else None      

    history, best, no_improve, start_epoch = [], {"macro_f1": -1.0, "epoch": -1}, 0, 0
    last_path, best_path = os.path.join(run_dir, "last.pt"), os.path.join(run_dir, "best.pt")
    if os.path.exists(last_path):
        ck = torch.load(last_path, map_location=device, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["optimizer"]); sched.load_state_dict(ck["scheduler"])
        history, best, no_improve, start_epoch = ck["history"], ck["best"], ck["no_improve"], ck["epoch"] + 1
        _set_rng_state(ck["rng"])
        if verbose:
            print(f"[resume] {run_dir} from epoch {start_epoch}")

    if calib_loader is None:
        raise ValueError("calib_loader (train utterances in fixed order) is required for uncertainty normalisation")
    if start_epoch == 0:
        model.calibrate_uncertainty(calib_loader, device)          

    status = "completed"
    for epoch in range(start_epoch, hp["max_epochs"]):
        t0 = time.time()
        train_loader.generator.manual_seed(seed * 10_000 + epoch)    
        model.train()
        use_aug = (hp.get("mod_dropout", 0.0) > 0 or hp.get("noise_aug_p", 0.0) > 0
                   or hp.get("utt_noise_p", 0.0) > 0 or hp.get("utt_mask_p", 0.0) > 0 or hp.get("feat_mask_rate", 0.0) > 0)
        if use_aug:
            if feature_std is None:
                raise ValueError("feature_std (compute_feature_std(train_dataset)) is required for augmentation")
            aug_gen = torch.Generator(); aug_gen.manual_seed(seed * 10_000 + epoch + 7)
        tot, tot_main, tot_aux, n_b = 0.0, 0.0, 0.0, 0
        for batch in train_loader:
            if use_aug:
                batch = augment_batch(batch, hp, feature_std, aug_gen)
            batch = batch_to_device(batch, device)
            out = model(batch)
            if loss_spec is None:
                loss, parts = compute_loss(out, batch, cw, hp["aux_weight"])
            else:
                loss, parts = imbalance_loss(out, batch, loss_spec, hp["aux_weight"])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), hp["grad_clip"])
            opt.step()
            tot += float(loss.detach()); tot_main += parts["loss_main"]; tot_aux += parts["loss_aux"]; n_b += 1

        calib = model.calibrate_uncertainty(calib_loader, device)      
        vm, varr = evaluate_model(model, val_loader, device, emotions, eval_seed=0)
        rho = float(hp.get("robust_rho", 0.0))
        deg = {}
        if rho > 0 and robust_loaders:
            for cname, ld in robust_loaders.items():
                deg[cname] = evaluate_model(model, ld, device, emotions, eval_seed=0)[0][hp.get("select_metric", "macro_f1")]
        score = robust_score(vm[hp.get("select_metric", "macro_f1")], deg, rho)     
        sched.step(score)
        rec = {"epoch": epoch, "train_loss": tot / n_b, "train_loss_main": tot_main / n_b, "train_loss_aux": tot_aux / n_b,
               "val_loss": vm["loss"], "val_accuracy": vm["accuracy"], "val_weighted_f1": vm["weighted_f1"],
               "val_macro_f1": vm["macro_f1"], "val_select_score": score, "val_degraded_macro_f1": deg, "val_f1_per_class": [vm["per_class"][e]["f1"] for e in emotions],
               "lr": opt.param_groups[0]["lr"], "seconds": time.time() - t0,
               "weights": weight_diagnostics(varr, model.modalities), "u_calibration": calib}
        history.append(rec)

        if score > best.get("score", best["macro_f1"]) + 1e-12:         
            best = {"macro_f1": vm["macro_f1"], "score": score, "epoch": epoch, "accuracy": vm["accuracy"], "weighted_f1": vm["weighted_f1"]}
            no_improve = 0
            _atomic_torch_save({"model": model.state_dict(), "epoch": epoch, "val_metrics": vm, "model_kwargs": model_kwargs,
                                "seed": seed, "fingerprint": fp}, best_path)
            save_predictions(os.path.join(run_dir, "val_predictions.csv"), varr, emotions, model.modalities)
        else:
            no_improve += 1

        _atomic_torch_save({"model": model.state_dict(), "optimizer": opt.state_dict(), "scheduler": sched.state_dict(),
                            "epoch": epoch, "history": history, "best": best, "no_improve": no_improve, "rng": _rng_state()},
                           last_path)
        with open(os.path.join(run_dir, "log.jsonl.tmp"), "w") as f:
            f.write("\n".join(json.dumps(r) for r in history) + "\n")
        os.replace(os.path.join(run_dir, "log.jsonl.tmp"), os.path.join(run_dir, "log.jsonl"))
        if verbose:
            print(f"  ep{epoch:02d} loss {rec['train_loss']:.3f} | val acc {vm['accuracy']:.3f} wF1 {vm['weighted_f1']:.3f} "
                  f"mF1 {vm['macro_f1']:.3f} {'*' if best['epoch'] == epoch else ' '} | w_shift {rec['weights']['w_shift_from_content']:.4f}")
        if no_improve >= hp["patience"]:
            status = "early_stopped"
            break
        if stop_after_epoch is not None and epoch + 1 >= stop_after_epoch and epoch + 1 < hp["max_epochs"]:
            return {"status": "interrupted", "epoch": epoch}

    result = {"status": status, "fingerprint": fp, "seed": seed, "best_epoch": best["epoch"],
              "best_val_macro_f1": best["macro_f1"], "best_val_accuracy": best["accuracy"],
              "best_val_weighted_f1": best["weighted_f1"], "best_select_score": best.get("score", best["macro_f1"]), "epochs_run": len(history),
              "final_weights": history[-1]["weights"], "completed": True}
    _atomic_json(result, res_path)
    return result


def load_best_model(run_dir, device):
    ck = torch.load(os.path.join(run_dir, "best.pt"), map_location=device, weights_only=False)
    model = ContextFusionModel(**ck["model_kwargs"]).to(device)
    model.load_state_dict(ck["model"])
    return model.eval(), ck


def aggregate_runs(records):
    """records: list of dicts with keys name, seed, best_val_macro_f1, best_val_accuracy, best_val_weighted_f1, best_epoch.
    Returns (summary DataFrame with mean +/- sample std over seeds, per-seed macro-F1 table)."""
    df = pd.DataFrame(records)
    rows = []
    for name, g in df.groupby("name", sort=False):
        d = {"name": name, "n_seeds": len(g)}
        for k in ("best_val_macro_f1", "best_val_accuracy", "best_val_weighted_f1", "best_epoch", "best_select_score"):
            if k not in g.columns:
                continue
            d[f"{k}_mean"] = float(g[k].mean())
            d[f"{k}_std"] = float(g[k].std(ddof=1)) if len(g) > 1 else float("nan")
        rows.append(d)
    return pd.DataFrame(rows), df.pivot(index="seed", columns="name", values="best_val_macro_f1")
'''

module_path = os.path.join(BASE_DIR, "meld_train.py")
with open(module_path, "w") as f:
    f.write(MODULE_SRC)
import meld_train
importlib.reload(meld_train)
from meld_train import (train_run, evaluate_model, load_best_model, aggregate_runs, compute_metrics, fingerprint, augment_batch, compute_feature_std,
                        prepare_loss, imbalance_loss)
from meld_models import class_weights_from_labels, load_split_dataset
from meld_context import make_loader
print(f"Wrote and imported {module_path}")

In [ ]:
df_train, train_ds = load_split_dataset(BASE_DIR, "train", IN_DIMS, EMOTIONS, context_config["speaker_mode"])
df_dev, dev_ds = load_split_dataset(BASE_DIR, "dev", IN_DIMS, EMOTIONS, context_config["speaker_mode"])
class_weights, class_counts = class_weights_from_labels(train_ds.labels, config["num_classes"])
print("class       count   weight")
for e, c, w in zip(EMOTIONS, class_counts.astype(int), class_weights.numpy()):
    print(f"{e:9s} {c:6d}   {w:.3f}")
with open(os.path.join(RESULTS_DIR, "class_weights.json"), "w") as f:
    json.dump({"emotions": EMOTIONS, "counts": class_counts.tolist(), "weights": class_weights.tolist()}, f, indent=2)

calib_loader = make_loader(train_ds, batch_size=32, shuffle=False, seed=0)    

FEATURE_STD = compute_feature_std(train_ds)          

from meld_train import corrupt_dataset
ROBUST_CONDS = [("missing_text", "missing", "text", None), ("dead_text", "dead", "text", None),
                ("noise_text_2", "noise", "text", 2.0), ("missing_audio", "missing", "audio", None)]
ROBUST_LOADERS = {n: make_loader(corrupt_dataset(dev_ds, k, m, lv, FEATURE_STD, seed=777 + i), batch_size=32, shuffle=False, seed=0)
                  for i, (n, k, m, lv) in enumerate(ROBUST_CONDS)} if ROBUST_RHO > 0 else None

def model_kwargs_for(fusion, lam):
    kw = {**MODEL_KWARGS, "fusion": fusion, "lam": float(lam)}
    if U_MEASURE:
        kw["u_measure"] = U_MEASURE
    if MC_SCOPE:
        kw["mc_scope"] = MC_SCOPE
    if U_HINGE:
        kw["u_hinge"] = True
    if U_NORM:
        kw["u_norm"] = U_NORM
    kw["mask_invalid"] = MASK_INVALID
    if DROPOUT:
        kw["dropout"] = float(DROPOUT)
    if U_CLIP > 0:
        kw["u_clip"] = U_CLIP
    return kw

In [ ]:
vresults = []
def check(name, cond, detail=""):
    vresults.append({"test": name, "passed": bool(cond), "detail": str(detail)})
    print(("PASS  " if cond else "FAIL  ") + name + (f"  [{detail}]" if detail else ""))

if RUN_VERIFICATION:
    VDIR = os.path.join(CKPT_DIR, "_verification")
    shutil.rmtree(VDIR, ignore_errors=True)
    vhp = {**{k: v for k, v in HP.items() if k != "select_metric"}, "max_epochs": 4, "patience": 10}   
    n_tr, n_dv = min(120, len(train_ds)), min(40, len(dev_ds))
    v_train, v_dev = Subset(train_ds, range(n_tr)), Subset(dev_ds, range(n_dv))
    mk_tl = lambda seed: make_loader(v_train, batch_size=16, shuffle=True, seed=seed)
    v_val = make_loader(v_dev, batch_size=16, shuffle=False, seed=0)
    v_calib = make_loader(v_train, batch_size=32, shuffle=False, seed=0)
    kw_prop = model_kwargs_for(FUSION_REL, 1.0)

    from meld_models import ContextFusionModel
    torch.manual_seed(0)
    m0 = ContextFusionModel(**kw_prop).to(device)
    full_val = make_loader(dev_ds, batch_size=32, shuffle=False, seed=0)
    met1, arr1 = evaluate_model(m0, full_val, device, EMOTIONS, eval_seed=0)
    met2, arr2 = evaluate_model(m0, full_val, device, EMOTIONS, eval_seed=0)
    from sklearn.metrics import f1_score, accuracy_score
    check("V1a evaluation covers every dev row exactly once, in CSV order", np.array_equal(arr1["row_idx"], np.arange(len(df_dev))))
    check("V1b y_true equals the CSV labels at those rows", np.array_equal(arr1["y_true"], dev_ds.labels[arr1["row_idx"]]))
    check("V1c macro-F1 / accuracy equal an independent sklearn computation",
          abs(met1["macro_f1"] - f1_score(arr1["y_true"], arr1["y_pred"], average="macro", labels=list(range(7)), zero_division=0)) < 1e-12
          and abs(met1["accuracy"] - accuracy_score(arr1["y_true"], arr1["y_pred"])) < 1e-12)
    check("V1d evaluation is deterministic (seeded MC passes)", met1["macro_f1"] == met2["macro_f1"] and np.array_equal(arr1["u"], arr2["u"]))

    r_b = train_run(os.path.join(VDIR, "B"), kw_prop, vhp, 42, mk_tl(42), v_val, class_weights, device, EMOTIONS, calib_loader=v_calib, feature_std=FEATURE_STD, verbose=False)
    log_b = [json.loads(l) for l in open(os.path.join(VDIR, "B", "log.jsonl"))]
    mf = [r["val_macro_f1"] for r in log_b]
    check("V2a best epoch = argmax of validation macro-F1 (first maximum)", r_b["best_epoch"] == int(np.argmax(mf)) and abs(r_b["best_val_macro_f1"] - max(mf)) < 1e-12,
          f"macro-F1 per epoch: {np.round(mf, 3).tolist()}")
    model_b, ck = load_best_model(os.path.join(VDIR, "B"), device)
    mb, _ = evaluate_model(model_b, v_val, device, EMOTIONS, eval_seed=0)
    check("V2b reloaded best.pt reproduces the logged validation macro-F1", abs(mb["macro_f1"] - r_b["best_val_macro_f1"]) < 1e-9 and ck["epoch"] == r_b["best_epoch"])
    need = ["config.json", "log.jsonl", "last.pt", "best.pt", "val_predictions.csv", "result.json"]
    check("V2c run directory contains all artifacts", all(os.path.exists(os.path.join(VDIR, "B", f)) for f in need))
    vp = pd.read_csv(os.path.join(VDIR, "B", "val_predictions.csv"))
    check("V2d saved predictions include per-utterance u and w for every modality",
          all(f"{p}_{m}" in vp.columns for p in ("u", "w", "wc") for m in ("text", "audio", "video")) and len(vp) == sum(l for _, l in dev_ds.blocks[:n_dv]))

    kA = os.path.join(VDIR, "A")
    r_int = train_run(kA, kw_prop, vhp, 42, mk_tl(42), v_val, class_weights, device, EMOTIONS, calib_loader=v_calib, feature_std=FEATURE_STD, stop_after_epoch=2, verbose=False)
    r_res = train_run(kA, kw_prop, vhp, 42, mk_tl(42), v_val, class_weights, device, EMOTIONS, calib_loader=v_calib, feature_std=FEATURE_STD, verbose=False)
    sa = torch.load(os.path.join(kA, "last.pt"), map_location="cpu", weights_only=False)["model"]
    sb = torch.load(os.path.join(VDIR, "B", "last.pt"), map_location="cpu", weights_only=False)["model"]
    wdiff = max((sa[k].float() - sb[k].float()).abs().max().item() for k in sb)
    log_a = [json.loads(l) for l in open(os.path.join(kA, "log.jsonl"))]
    mdiff = max(abs(x["val_macro_f1"] - y["val_macro_f1"]) for x, y in zip(log_a, log_b))
    check("V3 interrupted + resumed run equals an uninterrupted run",
          r_int["status"] == "interrupted" and len(log_a) == len(log_b) and wdiff < 1e-4 and mdiff < 1e-4,
          f"max weight diff={wdiff:.1e}, max macro-F1 diff={mdiff:.1e}")

    r_skip = train_run(os.path.join(VDIR, "B"), kw_prop, vhp, 42, mk_tl(42), v_val, class_weights, device, EMOTIONS, calib_loader=v_calib, feature_std=FEATURE_STD, verbose=False)
    try:
        train_run(os.path.join(VDIR, "B"), kw_prop, {**vhp, "lr": 1e-3}, 42, mk_tl(42), v_val, class_weights, device, EMOTIONS, calib_loader=v_calib, feature_std=FEATURE_STD, verbose=False)
        refused = False
    except RuntimeError:
        refused = True
    check("V4 completed run is skipped; different configuration is refused", r_skip["status"] == "skipped_completed" and refused)

    r_same = train_run(os.path.join(VDIR, "C"), kw_prop, vhp, 42, mk_tl(42), v_val, class_weights, device, EMOTIONS, calib_loader=v_calib, feature_std=FEATURE_STD, verbose=False)
    r_other = train_run(os.path.join(VDIR, "D"), kw_prop, vhp, 43, mk_tl(43), v_val, class_weights, device, EMOTIONS, calib_loader=v_calib, feature_std=FEATURE_STD, verbose=False)
    log_c = [json.loads(l) for l in open(os.path.join(VDIR, "C", "log.jsonl"))]
    log_d = [json.loads(l) for l in open(os.path.join(VDIR, "D", "log.jsonl"))]
    check("V5 same seed -> same training trajectory; different seed -> different",
          max(abs(x["train_loss"] - y["train_loss"]) for x, y in zip(log_c, log_b)) < 1e-4 and abs(log_d[0]["train_loss"] - log_b[0]["train_loss"]) > 1e-6)

    r_es = train_run(os.path.join(VDIR, "E"), model_kwargs_for(FUSION_BASE, 0.0), {**vhp, "max_epochs": 8, "patience": 1}, 42, mk_tl(42), v_val,
                     class_weights, device, EMOTIONS, calib_loader=v_calib, feature_std=FEATURE_STD, verbose=False)
    ok_es = (r_es["epochs_run"] == r_es["best_epoch"] + 2) if r_es["status"] == "early_stopped" else (r_es["best_epoch"] == r_es["epochs_run"] - 1)
    check("V6 early stopping stops patience epochs after the best validation macro-F1", ok_es, f"{r_es['status']}, best={r_es['best_epoch']}, ran={r_es['epochs_run']}")

    st_best = torch.load(os.path.join(VDIR, "B", "best.pt"), map_location=device, weights_only=False)["model"]
    model_b.calibrate_uncertainty(v_calib, device)
    check("V8 best.pt carries the train-set u statistics (re-calibration reproduces them; not the initial values)",
          bool(torch.allclose(model_b.u_mu, st_best["u_mu"], rtol=1e-6)) and bool(torch.allclose(model_b.u_sigma, st_best["u_sigma"], rtol=1e-6))
          and not bool(torch.allclose(st_best["u_mu"], torch.ones_like(st_best["u_mu"]))),
          f"u_mu={np.round(st_best['u_mu'].cpu().numpy(), 6)}")

    ab = next(iter(make_loader(train_ds, batch_size=64, shuffle=False, seed=0)))
    hp_aug = {"mod_dropout": 0.5, "noise_aug_p": 0.5, "noise_aug_levels": [0.5, 4.0]}
    g1 = torch.Generator(); g1.manual_seed(5); g2 = torch.Generator(); g2.manual_seed(5)
    a1 = augment_batch(copy.deepcopy(ab), hp_aug, FEATURE_STD, g1); a2 = augment_batch(copy.deepcopy(ab), hp_aug, FEATURE_STD, g2)
    kept = torch.stack([a1["valid"][m][:, 0] for m in a1["valid"]], -1)
    orig0 = torch.stack([ab["valid"][m][:, 0] for m in ab["valid"]], -1)          
    dropped0 = orig0 & ~kept                                                       
    noisy_any = [(a1["features"][m] - ab["features"][m]).abs().sum((1, 2)) > 0 for m in ab["features"]]
    noop = augment_batch(copy.deepcopy(ab), {"mod_dropout": 0.0, "noise_aug_p": 0.0}, FEATURE_STD, torch.Generator().manual_seed(1))
    d_txt = (a1["features"]["text"] - ab["features"]["text"])
    sel_ = noisy_any[0]
    ratio = float(d_txt[sel_].std() / FEATURE_STD["text"].mean()) if sel_.any() else float("nan")
    check("V9 augmentation: >=1 usable modality per dialogue, deterministic for a seed, no-op when disabled, noise only where not dropped",
          bool(((kept.sum(-1) >= 1) | (orig0.sum(-1) == 0)).all()) and all(torch.equal(a1["features"][m], a2["features"][m]) and torch.equal(a1["valid"][m], a2["valid"][m]) for m in a1["features"])
          and all(torch.equal(noop["features"][m], ab["features"][m]) and torch.equal(noop["valid"][m], ab["valid"][m]) for m in ab["features"])
          and all(not bool((noisy_any[j] & dropped0[:, j]).any()) for j in range(3)) and float(dropped0.float().mean()) > 0.2 and 0.5 < ratio < 4.5,
          f"dropped share={float(dropped0.float().mean()):.2f}, text-noise std/train-std={ratio:.2f}")

    vhp_aug = {**vhp, **hp_aug}
    rA = os.path.join(VDIR, "AUG_A"); rB = os.path.join(VDIR, "AUG_B")
    train_run(rA, kw_prop, vhp_aug, 42, mk_tl(42), v_val, class_weights, device, EMOTIONS, calib_loader=v_calib, feature_std=FEATURE_STD, stop_after_epoch=2, verbose=False)
    train_run(rA, kw_prop, vhp_aug, 42, mk_tl(42), v_val, class_weights, device, EMOTIONS, calib_loader=v_calib, feature_std=FEATURE_STD, verbose=False)
    train_run(rB, kw_prop, vhp_aug, 42, mk_tl(42), v_val, class_weights, device, EMOTIONS, calib_loader=v_calib, feature_std=FEATURE_STD, verbose=False)
    sa2 = torch.load(os.path.join(rA, "last.pt"), map_location="cpu", weights_only=False)["model"]
    sb2 = torch.load(os.path.join(rB, "last.pt"), map_location="cpu", weights_only=False)["model"]
    wd2 = max((sa2[k].float() - sb2[k].float()).abs().max().item() for k in sb2)
    check("V10 augmented training: interrupted + resumed run equals an uninterrupted run", wd2 < 1e-4, f"max weight diff={wd2:.1e}")

    def _aug(hp_, seed=3):
        g_ = torch.Generator(); g_.manual_seed(seed)
        return augment_batch(copy.deepcopy(ab), hp_, FEATURE_STD, g_)
    mods_ = list(ab["features"].keys()); real_ = ab["mask"]
    base_hp = {"mod_dropout": 0.3, "noise_aug_p": 0.2, "dead_aug_p": 0.15, "noise_aug_levels": [0.5, 4.0]}
    a_leg, a_zero = _aug(base_hp), _aug({**base_hp, "feat_mask_rate": 0.0, "utt_mask_p": 0.0, "utt_noise_p": 0.0})
    check("V12a new options set to 0 leave the legacy augmentation output bit-identical",
          all(torch.equal(a_leg["features"][m], a_zero["features"][m]) and torch.equal(a_leg["valid"][m], a_zero["valid"][m]) for m in mods_))
    x0 = ab["features"]["text"]; af, af2 = _aug({"feat_mask_rate": 0.2}), _aug({"feat_mask_rate": 0.2})
    zeroed = (af["features"]["text"] == 0) & (x0 != 0)
    frac = float(zeroed.float().sum() / (x0 != 0).float().sum())
    check("V12b feature masking zeroes ~rate of the elements, leaves the rest unchanged, is deterministic",
          abs(frac - 0.2) < 0.02 and bool(torch.equal(af["features"]["text"][~zeroed], x0[~zeroed])) and torch.equal(af["features"]["text"], af2["features"]["text"]),
          f"zeroed share={frac:.3f}")
    au = _aug({"utt_mask_p": 0.3})
    vs0 = torch.stack([ab["valid"][m] for m in mods_], -1); vs1 = torch.stack([au["valid"][m] for m in mods_], -1)
    rr = real_.unsqueeze(-1).expand_as(vs0)
    share = float((vs0 & ~vs1)[rr].float().mean())
    check("V12c utterance masking: never adds validity, keeps >=1 usable modality per usable utterance, rate near p, padding untouched",
          int((vs1 & ~vs0).sum()) == 0 and bool((vs1.any(-1) | ~vs0.any(-1) | ~real_).all()) and 0.15 < share < 0.32 and bool(torch.equal(vs1[~rr], vs0[~rr])),
          f"newly flagged share={share:.3f}")
    an = _aug({"utt_noise_p": 0.3, "noise_aug_levels": [0.5, 4.0]})
    ch = [((an["features"][m] - ab["features"][m]).abs().sum(-1) > 0) for m in mods_]
    only_usable = all(int((c & ~(ab["valid"][m] & real_)).sum()) == 0 for c, m in zip(ch, mods_))
    rate = float(torch.stack(ch, -1)[rr].float().mean())
    zt = (an["features"]["text"] - ab["features"]["text"])[ch[0]] / FEATURE_STD["text"].view(1, -1)
    rms = float((zt ** 2).mean().sqrt())
    check("V12d adaptive utterance noise: only usable real utterances change, rate near p, RMS level matches log-uniform(0.5, 4) = 1.95",
          only_usable and 0.2 < rate < 0.33 and 1.6 < rms < 2.3, f"rate={rate:.3f}, rms level={rms:.2f}")
    hp_b = {**vhp, **base_hp, "feat_mask_rate": 0.1, "utt_mask_p": 0.15, "utt_noise_p": 0.15}
    rA2, rB2 = os.path.join(VDIR, "AUGB_A"), os.path.join(VDIR, "AUGB_B")
    train_run(rA2, kw_prop, hp_b, 42, mk_tl(42), v_val, class_weights, device, EMOTIONS, calib_loader=v_calib, feature_std=FEATURE_STD, stop_after_epoch=2, verbose=False)
    train_run(rA2, kw_prop, hp_b, 42, mk_tl(42), v_val, class_weights, device, EMOTIONS, calib_loader=v_calib, feature_std=FEATURE_STD, verbose=False)
    train_run(rB2, kw_prop, hp_b, 42, mk_tl(42), v_val, class_weights, device, EMOTIONS, calib_loader=v_calib, feature_std=FEATURE_STD, verbose=False)
    sa3 = torch.load(os.path.join(rA2, "last.pt"), map_location="cpu", weights_only=False)["model"]
    sb3 = torch.load(os.path.join(rB2, "last.pt"), map_location="cpu", weights_only=False)["model"]
    wd3 = max((sa3[k].float() - sb3[k].float()).abs().max().item() for k in sb3)
    check("V12e training with the new augmentations: interrupted + resumed run equals an uninterrupted run", wd3 < 1e-4, f"max weight diff={wd3:.1e}")

    recs = [{"name": "x", "seed": s, "best_val_macro_f1": v, "best_val_accuracy": v, "best_val_weighted_f1": v, "best_epoch": 1}
            for s, v in zip((1, 2, 3), (0.2, 0.4, 0.6))]
    summ, piv = aggregate_runs(recs)
    check("V7 aggregate_runs mean/std", abs(summ.loc[0, "best_val_macro_f1_mean"] - 0.4) < 1e-12 and abs(summ.loc[0, "best_val_macro_f1_std"] - 0.2) < 1e-12)

    from meld_models import compute_loss as _compute_loss
    _g = torch.Generator().manual_seed(11)
    _B, _L, _C = 3, 6, 7
    _mk = lambda: torch.randn(_B, _L, _C, generator=_g, requires_grad=True)
    _mask = torch.rand(_B, _L, generator=_g) > 0.3
    _mask[:, 0] = True
    _fo = {"logits": _mk(), "modality_logits": {m: _mk() for m in ("text", "audio", "video")}}
    _fb = {"mask": _mask, "labels": torch.where(_mask, torch.randint(0, _C, (_B, _L), generator=_g), torch.full((_B, _L), -100))}
    _cw = class_weights.detach().cpu()
    _ref = float(_compute_loss(_fo, _fb, _cw, 0.5)[0])
    _spec = lambda cfg: prepare_loss(_cw, cfg)
    _l_wce = float(imbalance_loss(_fo, _fb, _spec({"kind": "wce"}), 0.5)[0])
    check("V11a kind='wce' reproduces meld_models.compute_loss exactly", abs(_l_wce - _ref) < 1e-6, f"{_l_wce:.6f} vs {_ref:.6f}")
    _l_f0 = float(imbalance_loss(_fo, _fb, _spec({"kind": "focal", "gamma": 0.0, "weight_power": 1.0}), 0.5)[0])
    check("V11b focal with gamma=0 and full weights equals weighted CE", abs(_l_f0 - _ref) < 1e-5, f"{_l_f0:.6f} vs {_ref:.6f}")
    _l_f2 = float(imbalance_loss(_fo, _fb, _spec({"kind": "focal", "gamma": 2.0, "weight_power": 1.0}), 0.5)[0])
    check("V11c focal (gamma=2) is never larger than weighted CE with the same weights", _l_f2 < _l_f0, f"{_l_f2:.6f} < {_l_f0:.6f}")
    _l_u = float(imbalance_loss(_fo, _fb, _spec({"kind": "wce_soft", "weight_power": 0.0}), 0.5)[0])
    _yy = _fb["labels"][_fb["mask"]]
    _un = float(torch.nn.functional.cross_entropy(_fo["logits"][_mask], _yy) + 0.5 * torch.stack([torch.nn.functional.cross_entropy(_fo["modality_logits"][m][_mask], _yy) for m in _fo["modality_logits"]]).mean())
    check("V11d wce_soft with power 0 equals unweighted CE", abs(_l_u - _un) < 1e-6, f"{_l_u:.6f} vs {_un:.6f}")
    _l_la0 = float(imbalance_loss(_fo, _fb, _spec({"kind": "logit_adj", "tau": 0.0}), 0.5)[0])
    _l_la1 = float(imbalance_loss(_fo, _fb, _spec({"kind": "logit_adj", "tau": 1.0}), 0.5)[0])
    check("V11e logit_adj with tau=0 equals unweighted CE, tau=1 differs", abs(_l_la0 - _un) < 1e-6 and abs(_l_la1 - _un) > 1e-3, f"{_l_la0:.6f}, {_l_la1:.6f}")
    _sp = _spec({"kind": "wce_soft", "weight_power": 0.5})
    _pi = (1.0 / _cw) / (1.0 / _cw).sum()
    _order_ok = bool(torch.equal(torch.argsort(_sp["weights"]), torch.argsort(_cw)))
    check("V11f soft weights keep the class ordering and the overall loss scale (sum pi*w = 1), and are milder than the full weights",
          _order_ok and abs(float((_sp["weights"] * _pi).sum()) - 1.0) < 1e-5 and float(_sp["weights"].max() / _sp["weights"].min()) < float(_cw.max() / _cw.min()))
    _gok = True
    for _cfg in ({"kind": "wce"}, {"kind": "wce_soft"}, {"kind": "focal"}, {"kind": "logit_adj"}):
        for _t in [_fo["logits"]] + list(_fo["modality_logits"].values()):
            _t.grad = None
        imbalance_loss(_fo, _fb, _spec(_cfg), 0.5)[0].backward()
        _gok &= bool(torch.isfinite(_fo["logits"].grad).all()) and float(_fo["logits"].grad.abs().sum()) > 0 \
            and all(float(t.grad.abs().sum()) > 0 for t in _fo["modality_logits"].values())
    check("V11g every loss gives finite, non-zero gradients to the fused logits and every modality head", _gok)
    _fo2 = {"logits": _fo["logits"].detach().clone(), "modality_logits": {m: t.detach().clone() for m, t in _fo["modality_logits"].items()}}
    _fo2["logits"][~_mask] += 100.0
    for _t in _fo2["modality_logits"].values():
        _t[~_mask] -= 50.0
    _same = all(abs(float(imbalance_loss(_fo, _fb, _spec(c), 0.5)[0]) - float(imbalance_loss(_fo2, _fb, _spec(c), 0.5)[0])) < 1e-5
                for c in ({"kind": "wce"}, {"kind": "focal"}, {"kind": "logit_adj"}))
    check("V11h padded positions (mask False) do not influence any loss", _same)

    failed = [r for r in vresults if not r["passed"]]
    with open(os.path.join(RESULTS_DIR, "training_verification.json"), "w") as f:
        json.dump({"n_tests": len(vresults), "n_failed": len(failed), "results": vresults}, f, indent=2)
    assert not failed, f"{len(failed)} verification test(s) failed: {[r['test'] for r in failed]}"
    shutil.rmtree(VDIR, ignore_errors=True)
    print(f"\nAll {len(vresults)} training-machinery checks passed.")
else:
    print("Verification skipped (MELD_P9_VERIFY=0).")

In [ ]:

EXPERIMENTS = [("baseline", FUSION_BASE, 0.0)] + [(f"proposed_lam{lam:g}", FUSION_REL, lam) for lam in LAMBDA_GRID]
import hashlib
FEATURE_DESC = {k: config.get(k) for k in ("audio_feature_set", "audio_model", "video_backbone", "video_hsemotion_model")}
if config.get("text_feature_set"):
    FEATURE_DESC["text_feature_set"] = config["text_feature_set"]       
if config.get("audio_ft_feature_set"):
    FEATURE_DESC["audio_ft_feature_set"] = config["audio_ft_feature_set"]   
LEGACY_FEATURES = config.get("audio_feature_set") == "eGeMAPSv02" and config.get("video_backbone") == "densenet121"
FEATURE_TAG = "" if LEGACY_FEATURES else "_feat" + hashlib.sha1(json.dumps(FEATURE_DESC, sort_keys=True).encode()).hexdigest()[:8]
PHASE9_DIR = os.path.join(CKPT_DIR, ("phase9" if FAMILY == "legacy" else f"phase9_{FAMILY}") + FEATURE_TAG + LOSS_TAG + AUG_TAG + HP_TAG + SEL_TAG)  
print(f"Run folder: {PHASE9_DIR}  (feature set: {FEATURE_DESC})")
records = []
t_start = time.time()
for seed in SEEDS:                                    
    for name, fusion, lam in EXPERIMENTS:
        run_dir = os.path.join(PHASE9_DIR, name, f"seed{seed}")
        print(f"\n=== {name} | seed {seed} ===")
        train_loader = make_loader(train_ds, batch_size=BATCH_SIZE, shuffle=True, seed=seed)
        val_loader = make_loader(dev_ds, batch_size=32, shuffle=False, seed=0)
        res = train_run(run_dir, model_kwargs_for(fusion, lam), HP, seed, train_loader, val_loader, class_weights, device, EMOTIONS,
                        calib_loader=calib_loader, feature_std=FEATURE_STD, robust_loaders=ROBUST_LOADERS)
        records.append({"name": name, "seed": seed, "run_dir": run_dir, **{k: res[k] for k in
                        ("best_val_macro_f1", "best_val_accuracy", "best_val_weighted_f1", "best_select_score", "best_epoch", "epochs_run", "status")}})
print(f"\nAll runs finished in {(time.time() - t_start) / 60:.1f} min")

In [ ]:
summary, per_seed = aggregate_runs(records)
pd.set_option("display.width", 200)
cols = ["name", "n_seeds", "best_val_macro_f1_mean", "best_val_macro_f1_std", "best_val_accuracy_mean", "best_val_weighted_f1_mean", "best_epoch_mean"]
print(summary[cols].round(4).to_string(index=False))
print("\nPer-seed validation macro-F1:")
print(per_seed.round(4).to_string())
summary.to_csv(os.path.join(OUT_DIR, "phase9_val_summary.csv"), index=False)
per_seed.to_csv(os.path.join(OUT_DIR, "phase9_val_per_seed.csv"))
pd.DataFrame(records).pivot(index="seed", columns="name", values="best_val_accuracy").to_csv(os.path.join(OUT_DIR, "phase9_val_per_seed_accuracy.csv"))   

prop = summary[summary["name"].str.startswith("proposed")]
SEL_COL = "best_select_score_mean" if "best_select_score_mean" in prop.columns else "best_val_macro_f1_mean"   
best_row = prop.loc[prop[SEL_COL].idxmax()]
selected = {"selected_experiment": best_row["name"], "lambda": float(best_row["name"].replace("proposed_lam", "")),
            "val_macro_f1_mean": float(best_row["best_val_macro_f1_mean"]), "val_select_score_mean": float(best_row[SEL_COL]),
            "criterion": ("mean robust validation score over seeds (rho=%g)" % ROBUST_RHO) if SEL_COL.startswith("best_select") else "mean validation macro-F1 over seeds"}
if "baseline" in per_seed.columns and best_row["name"] in per_seed.columns:
    diff = per_seed[best_row["name"]] - per_seed["baseline"]
    selected.update({"paired_val_macro_f1_diff_mean": float(diff.mean()),
                     "paired_val_macro_f1_diff_std": float(diff.std(ddof=1)) if len(diff) > 1 else None,
                     "seeds_where_proposed_higher": int((diff > 0).sum()), "n_seeds": int(len(diff))})
with open(os.path.join(OUT_DIR, "phase9_selected_lambda.json"), "w") as f:
    json.dump(selected, f, indent=2)
print("\nSelected on validation:", json.dumps(selected, indent=2))

diag_rows = []
for r in records:
    log = [json.loads(l) for l in open(os.path.join(r["run_dir"], "log.jsonl"))]
    wd = log[r["best_epoch"]]["weights"]
    diag_rows.append({"name": r["name"], "seed": r["seed"], "w_shift_from_content": wd["w_shift_from_content"], "w_spread": wd["w_spread"],
                      **{f"uraw_{m}": v for m, v in wd["u_raw_mean"].items()}, **{f"u_{m}": v for m, v in wd["u_mean"].items()}, **{f"w_{m}": v for m, v in wd["w_mean"].items()}})
diag = pd.DataFrame(diag_rows)
diag.to_csv(os.path.join(OUT_DIR, "phase9_weight_diagnostics.csv"), index=False)
print("\nReliability diagnostics (mean over seeds, best epoch). u = normalised uncertainty, uraw = raw; w_shift = total-variation distance between final weights and content-gate weights:")
print(diag.groupby("name", sort=False).mean(numeric_only=True).drop(columns="seed").to_string(float_format=lambda x: f"{x:.6f}"))
with open(os.path.join(OUT_DIR, "phase9_run_index.json"), "w") as f:
    json.dump(records, f, indent=2)
with open(os.path.join(OUT_DIR, "phase9_config.json"), "w") as f:
    json.dump({"seeds": SEEDS, "lambda_grid": LAMBDA_GRID, "hp": HP, "batch_size": BATCH_SIZE, "model_kwargs": MODEL_KWARGS,
               "family": FAMILY, "select_metric": SELECT_METRIC, "feature_set": FEATURE_DESC, "run_root": PHASE9_DIR, "fusion_baseline": FUSION_BASE, "fusion_proposed": FUSION_REL,
               "overrides": {"u_measure": U_MEASURE or None, "mc_scope": MC_SCOPE or None, "u_hinge": U_HINGE, "u_norm": U_NORM or None, "u_clip": U_CLIP, "mask_invalid": MASK_INVALID,
                             "mod_dropout": MOD_DROPOUT, "noise_aug_p": NOISE_AUG_P, "dead_aug_p": DEAD_AUG_P, "robust_rho": ROBUST_RHO, "dropout": float(DROPOUT) if DROPOUT else None},
               "class_weights": class_weights.tolist(), "selection_metric": "validation macro-F1", "loss_selection": LOSS_SELECTION}, f, indent=2)

In [ ]:
for r in records:
    for fn in ("config.json", "log.jsonl", "best.pt", "last.pt", "val_predictions.csv", "result.json"):
        assert os.path.exists(os.path.join(r["run_dir"], fn)), f"Missing {fn} in {r['run_dir']}"
assert len(records) == len(SEEDS) * len(EXPERIMENTS)
print(f"{len(records)} runs verified ({len(SEEDS)} seeds x {len(EXPERIMENTS)} models).")
print("Phase 9 Model Training completed successfully.")